# Servo J sweep 결과 플롯

`run_servo_j_parameter_sweep.py`가 저장한 `summary.csv`, `*_profile.csv`, `*_commands.csv`를 읽어서 trial별로 확인합니다.

In [1]:
from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12})

## 결과 폴더 선택

`RESULT_DIR = None`이면 `servo_j_sweep_results` 아래 최신 폴더를 자동으로 선택합니다.

In [ ]:
ROOT_DIR = Path("..").resolve()
SWEEP_ROOT = ROOT_DIR / "servo_j_sweep_results"
RESULT_DIR = None

def latest_result_dir(root):
    dirs = [p for p in root.iterdir() if p.is_dir() and (p / "summary.csv").exists()]
    if not dirs:
        raise FileNotFoundError(f"No summary.csv found under {root}")
    return sorted(dirs)[-1]

result_dir = Path(RESULT_DIR).resolve() if RESULT_DIR else latest_result_dir(SWEEP_ROOT)
summary_path = result_dir / "summary.csv"

print(result_dir)
print(summary_path)

## 로드 함수

In [ ]:
def to_float(value):
    if value is None or value == "":
        return None
    return float(value)


def read_rows(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def numeric_column(rows, key):
    return np.array([float(row[key]) for row in rows], dtype=float)


def load_summary(path):
    rows = read_rows(path)
    for row in rows:
        for key in [
            "trial", "joint", "gain", "t1_s", "t2_s", "alpha", "servo_dt_s",
            "start_deg", "end_deg", "samples", "commands", "last_command_time_s",
            "final_settle_time_s", "final_settle_dt_from_last_cmd_s", "final_ref_deg",
            "max_ref_deg", "min_ref_deg", "crossing_time_s", "crossing_interval_start_s",
            "crossing_interval_end_s", "crossing_interval_start_deg", "crossing_interval_end_deg",
            "crossing_dt_from_last_cmd_s", "reset_max_ref_err_deg",
        ]:
            if key in row:
                row[key] = to_float(row[key])
    return rows


summary = load_summary(summary_path)
print(f"trials: {len(summary)}")

## Trial 목록

In [ ]:
for row in summary:
    print(
        f"[{int(row['trial']):03d}] "
        f"gain={row['gain']:g}, t2={row['t2_s']:g}, alpha={row['alpha']:g}, "
        f"cross={row.get('crossing_time_s')}, final={row.get('final_settle_time_s')}"
    )

## 단일 Trial 플롯

`TRIAL_INDEX`를 바꿔가며 실행하면 하나씩 확인할 수 있습니다.

In [ ]:
TRIAL_INDEX = 1


def get_trial(summary_rows, trial_index):
    for row in summary_rows:
        if int(row["trial"]) == int(trial_index):
            return row
    raise KeyError(f"trial {trial_index} not found")


def plot_trial(trial_index):
    row = get_trial(summary, trial_index)
    profile_path = result_dir / row["profile_csv"]
    command_path = result_dir / row["command_csv"]

    profile = read_rows(profile_path)
    commands = read_rows(command_path)

    t = numeric_column(profile, "t_s")
    q_ref = numeric_column(profile, "jnt_ref_deg")
    dq = numeric_column(profile, "angle_change_deg")
    cmd_t = numeric_column(commands, "t_s")
    cmd_q = numeric_column(commands, "cmd_deg")
    v_ref = np.gradient(q_ref, t) if len(t) >= 2 else np.zeros_like(q_ref)

    target = row["end_deg"]
    last_cmd = row.get("last_command_time_s")
    crossing = row.get("crossing_time_s")
    final_settle = row.get("final_settle_time_s")

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), facecolor="white", dpi=150, sharex=True)

    axes[0].plot(t, q_ref, label="jnt_ref", linewidth=1.6)
    axes[0].plot(cmd_t, cmd_q, label="sent cmd", linewidth=1.0, alpha=0.75, drawstyle="steps-post")
    axes[0].axhline(target, color="tab:red", linestyle=":", linewidth=1.2, label=f"target {target:g} deg")
    axes[0].set_ylabel("angle (deg)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc="best")

    axes[1].plot(t, dq, color="tab:blue", linewidth=1.5, label="angle change")
    axes[1].set_ylabel("change (deg)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="best")

    axes[2].plot(t, v_ref, color="tab:purple", linewidth=1.5, label="d(jnt_ref)/dt")
    axes[2].set_xlabel("time (s)")
    axes[2].set_ylabel("velocity (deg/s)")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc="best")

    markers = [
        (last_cmd, "tab:orange", ":", "last cmd"),
        (crossing, "tab:cyan", "--", "target crossing"),
        (final_settle, "tab:green", "-.", "final settle"),
    ]
    for marker_t, color, linestyle, label in markers:
        if marker_t is None:
            continue
        for ax in axes:
            ax.axvline(marker_t, color=color, linestyle=linestyle, linewidth=1.2, label=label)

    if crossing is not None:
        cross_idx = int(np.clip(np.searchsorted(t, crossing), 0, len(t) - 1))
        axes[0].scatter([crossing], [q_ref[cross_idx]], color="tab:cyan", edgecolor="black", zorder=5)
        dt_text = "" if last_cmd is None else f"\nΔt from last cmd: {(crossing - last_cmd) * 1000:.1f} ms"
        axes[0].annotate(
            f"cross {crossing:.3f} s{dt_text}",
            xy=(crossing, q_ref[cross_idx]),
            xytext=(10, -50),
            textcoords="offset points",
            color="tab:cyan",
            bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "tab:cyan", "alpha": 0.85},
            arrowprops={"arrowstyle": "->", "color": "tab:cyan", "linewidth": 0.8},
        )

    if final_settle is not None:
        final_idx = int(np.clip(np.searchsorted(t, final_settle), 0, len(t) - 1))
        axes[0].scatter([final_settle], [q_ref[final_idx]], color="tab:green", edgecolor="black", zorder=5)

    title = (
        f"trial {int(row['trial']):03d}: "
        f"gain={row['gain']:g}, t1={row['t1_s']:g}, t2={row['t2_s']:g}, alpha={row['alpha']:g}"
    )
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

    print(f"profile: {profile_path}")
    print(f"commands: {command_path}")
    print(f"last_cmd={last_cmd}, crossing={crossing}, final_settle={final_settle}")
    print(f"final_ref={row.get('final_ref_deg')}, max_ref={row.get('max_ref_deg')}, reset_err={row.get('reset_max_ref_err_deg')}")


plot_trial(TRIAL_INDEX)

## 여러 Trial 연속 출력

아래 셀의 범위를 조정해서 여러 trial을 연속으로 확인할 수 있습니다.

In [ ]:
PLOT_TRIALS = []  # 예: [1, 2, 3]

for trial_index in PLOT_TRIALS:
    plot_trial(trial_index)